In [ ]:
"""
Pseudo-Correlation Flip Test for HEART dataset (N≤5000 per model)
- All models (including LR, RF, MLP) are limited to at most 5000 training samples
- TabPFN & FT-TabPFN use 5000 in-context examples
- Fair comparison under identical data budget
"""

import pandas as pd
import numpy as np
import torch
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
import fairlearn.metrics as fm
from tabpfn import TabPFNClassifier
from tabpfn.utils import meta_dataset_collator
from tabpfn.finetune_utils import clone_model_for_evaluation
import warnings
warnings.filterwarnings("ignore")

# ==================== CONFIG ====================
MAX_TRAIN_SAMPLES = 5000  # 所有模型统一数据预算
INFERENCE_CONTEXT_SAMPLES = 5000

config = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "n_inference_context_samples": INFERENCE_CONTEXT_SAMPLES,
    "finetuning": {
        "epochs": 10,
        "learning_rate": 1e-5,
        "meta_batch_size": 1,
        "batch_size": 128
    }
}

# ==================== HELPERS ====================
def to_numpy_safe(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.array(x)

def limit_train_size(X_train, y_train, sensitive_dict, max_samples=MAX_TRAIN_SAMPLES, seed=42):
    """Sample at most max_samples from training data (with replacement if needed)"""
    np.random.seed(seed)
    n_orig = len(X_train)
    if n_orig <= max_samples:
        indices = np.random.choice(n_orig, max_samples, replace=True)
    else:
        indices = np.random.choice(n_orig, max_samples, replace=False)

    X_limited = X_train.iloc[indices].reset_index(drop=True)
    y_limited = y_train[indices]

    sensitive_limited = {}
    for name, series in sensitive_dict.items():
        sensitive_limited[name] = series.iloc[indices].reset_index(drop=True)

    print(f"[Data Limit] Train size: {n_orig} → {len(X_limited)} (budget={max_samples})")
    return X_limited, y_limited, sensitive_limited

def add_z_spur(X, y, correlation='positive'):
    z_spur = np.where(y == 1, np.random.normal(1.0, 0.5, len(y)),
                               np.random.normal(0.0, 0.5, len(y)))
    if correlation == 'negative':
        z_spur = -z_spur
    X_out = X.copy()
    X_out['Z_spur'] = z_spur
    return X_out

def get_preprocessor_with_z(num_cols, cat_cols):
    return ColumnTransformer([
        ('num', StandardScaler(), num_cols + ['Z_spur']),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ])

def evaluate_model(model, X_train, y_train, X_test, y_test, sensitive_dict,
                   is_tabpfn=False, preprocessor=None, eval_cfg=None):
    if is_tabpfn:
        eval_model = clone_model_for_evaluation(model, eval_cfg or {}, TabPFNClassifier)
        eval_model.fit(X_train.values if hasattr(X_train, 'values') else X_train, y_train)
        preds = eval_model.predict(X_test.values if hasattr(X_test, 'values') else X_test)
        probs = eval_model.predict_proba(X_test.values if hasattr(X_test, 'values') else X_test)
        probs = probs[:, 1] if probs.shape[1] > 1 else probs[:, 0]
    else:
        X_tr = preprocessor.transform(X_train)
        X_te = preprocessor.transform(X_test)
        model.fit(X_tr, y_train)
        preds = model.predict(X_te)
        probs = model.predict_proba(X_te)[:, 1]

    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)

    fairness = {'Sensitive': [], 'DP': [], 'EO': []}
    for name, s_feat in sensitive_dict.items():
        dp = fm.demographic_parity_difference(y_test, preds, sensitive_features=s_feat)
        eo = fm.equalized_odds_difference(y_test, preds, sensitive_features=s_feat)
        fairness['Sensitive'].append(name)
        fairness['DP'].append(round(dp, 4))
        fairness['EO'].append(round(eo, 4))

    return acc, auc, pd.DataFrame(fairness), preds

# ==================== MAIN EXPERIMENT ====================
def run_heart_pseudo_correlation_flip():
    df = pd.read_csv("heart.csv")
    df["age_group"] = (df["age"] >= 50).map({True: ">=50", False: "<50"})

    target = "target"
    sensitive_feats = {"sex": df["sex"], "age_group": df["age_group"]}

    X = df.drop(columns=[target])
    y = df[target].values

    num_cols = X.select_dtypes('number').columns.tolist()
    cat_cols = X.select_dtypes('object').columns.tolist()

    results = []

    for run in range(1, 6):
        seed = 41 + run
        np.random.seed(seed)
        torch.manual_seed(seed)
        print(f"\n{'='*25} RUN {run}/5 (seed={seed}) {'='*25}")

        # Split
        X_train_full, X_test, y_train_full, y_test = train_test_split(
            X, y, test_size=0.3, random_state=seed, stratify=y
        )
        s_test = {k: v.iloc[X_test.index] for k, v in sensitive_feats.items()}

        # Limit all models to 5000 training samples
        X_train, y_train, s_train = limit_train_size(
            X_train_full, y_train_full,
            {k: v.iloc[X_train_full.index] for k, v in sensitive_feats.items()},
            max_samples=MAX_TRAIN_SAMPLES, seed=seed
        )

        # Add spurious feature
        X_train_z = add_z_spur(X_train, y_train, 'positive')
        X_test_flip = add_z_spur(X_test, y_test, 'negative')
        X_test_normal = add_z_spur(X_test, y_test, 'positive')

        preprocessor = get_preprocessor_with_z(num_cols, cat_cols)
        X_train_prep = preprocessor.fit_transform(X_train_z)

        # Models (all see same limited data)
        models = {
            'LR': Pipeline([('clf', LogisticRegression(C=0.1, max_iter=1000))]),
            'RF': Pipeline([('clf', RandomForestClassifier(n_estimators=50, max_depth=5))]),
            'MLP': Pipeline([('clf', MLPClassifier(hidden_layer_sizes=(50,), max_iter=300, alpha=0.01))]),
            'TabPFN': TabPFNClassifier(device=config["device"], n_estimators=8),
            'FT-TabPFN': None
        }

        # === Finetune FT-TabPFN ===
        print("Finetuning FT-TabPFN...")
        try:
            clf_cfg = {
                "ignore_pretraining_limits": True,
                "device": config["device"],
                "n_estimators": 1,
                "inference_precision": torch.float32,
            }
            ft_clf = TabPFNClassifier(**clf_cfg, fit_mode="batched", differentiable_input=False)
            ft_clf._initialize_model_variables()

            datasets = ft_clf.get_preprocessed_datasets(
                X_train_prep, y_train,
                partial(train_test_split, test_size=0.3, random_state=seed),
                batch_size=128
            )
            loader = DataLoader(datasets, batch_size=1, collate_fn=meta_dataset_collator, shuffle=True)
            opt = Adam(ft_clf.models_[0].parameters(), lr=config["finetuning"]["learning_rate"])

            for epoch in range(config["finetuning"]["epochs"]):
                for batch in tqdm(loader, desc=f"FT Epoch {epoch+1}", leave=False):
                    X_tr, X_val, y_tr, y_val, cat, conf = batch
                    if len(np.unique(to_numpy_safe(y_tr))) < 2: continue
                    opt.zero_grad()
                    ft_clf.fit_from_preprocessed(X_tr, y_tr, cat, conf)
                    logits = ft_clf.forward(X_val, return_logits=True)
                    loss = torch.nn.CrossEntropyLoss()(logits, y_val.to(config["device"]))
                    loss.backward()
                    opt.step()
            models['FT-TabPFN'] = ft_clf
        except Exception as e:
            print(f"[FT Error] {e}")

        eval_cfg = {"inference_config": {"SUBSAMPLE_SAMPLES": INFERENCE_CONTEXT_SAMPLES}}

        # === Evaluate all models ===
        for name, model in models.items():
            if model is None: continue

            # Flipped test (negative correlation)
            acc_f, auc_f, fair_f, preds_f = evaluate_model(
                model, X_train_z, y_train, X_test_flip, y_test, s_test,
                is_tabpfn=name.startswith('TabPFN'),
                preprocessor=preprocessor if not name.startswith('TabPFN') else None,
                eval_cfg=eval_cfg if name == 'FT-TabPFN' else {}
            )
            # Normal test (positive correlation)
            acc_n, auc_n, fair_n, preds_n = evaluate_model(
                model, X_train_z, y_train, X_test_normal, y_test, s_test,
                is_tabpfn=name.startswith('TabPFN'),
                preprocessor=preprocessor if not name.startswith('TabPFN') else None,
                eval_cfg=eval_cfg if name == 'FT-TabPFN' else {}
            )

            consistency = np.mean(preds_f == preds_n)
            delta_acc = acc_f - acc_n

            for _, row in fair_f.iterrows():
                s = row['Sensitive']
                dp_f = row['DP']
                eo_f = row['EO']
                dp_n = fair_n.loc[fair_n['Sensitive'] == s, 'DP'].iloc[0]
                eo_n = fair_n.loc[fair_n['Sensitive'] == s, 'EO'].iloc[0]

                results.append({
                    'Run': run, 'Model': name, 'Sensitive': s,
                    'Accuracy_Flip': acc_f, 'Accuracy_Normal': acc_n, 'ΔAccuracy': delta_acc,
                    'AUC_Flip': auc_f, 'AUC_Normal': auc_n,
                    'DP_Flip': dp_f, 'DP_Normal': dp_n, 'ΔDP': dp_f - dp_n,
                    'EO_Flip': eo_f, 'EO_Normal': eo_n, 'ΔEO': eo_f - eo_n,
                    'Consistency': consistency
                })

    # === Final Table ===
    df_res = pd.DataFrame(results)
    agg = df_res.groupby(['Model', 'Sensitive']).agg({
        'ΔAccuracy': ['mean', 'std'],
        'ΔDP': ['mean', 'std'],
        'ΔEO': ['mean', 'std'],
        'Consistency': ['mean', 'std'],
        'Accuracy_Normal': 'mean',
        'AUC_Normal': 'mean'
    }).round(4)

    agg.columns = ['ΔAcc_mean', 'ΔAcc_std', 'ΔDP_mean', 'ΔDP_std',
                   'ΔEO_mean', 'ΔEO_std', 'Consistency_mean', 'Consistency_std',
                   'Acc_Base', 'AUC_Base']

    print("\n" + "="*120)
    print("FINAL RESULTS: All Models Limited to ≤5000 Training Samples")
    print("="*120)
    print(agg.to_markdown())

    return agg

# ==================== RUN ====================
if __name__ == "__main__":
    run_heart_pseudo_correlation_flip()

In [ ]:
"""
Pseudo-Correlation Flip Test for Bank Personal Loan Dataset (N≤5000 per model)
- All models (including LR, RF, MLP) are limited to at most 5000 training samples
- TabPFN & FT-TabPFN use 5000 in-context examples
- Fair comparison under identical data budget
- Sensitive attributes: Education, Family
"""

import pandas as pd
import numpy as np
import torch
from torch.optim import Adam
from torch.utils.data import DataLoader
from functools import partial
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
import fairlearn.metrics as fm
from tabpfn import TabPFNClassifier
from tabpfn.utils import meta_dataset_collator
from tabpfn.finetune_utils import clone_model_for_evaluation
import warnings
warnings.filterwarnings("ignore")

# ==================== CONFIG ====================
MAX_TRAIN_SAMPLES = 5000          # 所有模型统一上限
INFERENCE_CONTEXT_SAMPLES = 5000  # TabPFN 推理时强制 5000 条 context

config = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "n_inference_context_samples": INFERENCE_CONTEXT_SAMPLES,
    "finetuning": {
        "epochs": 10,
        "learning_rate": 1e-5,
        "meta_batch_size": 1,
        "batch_size": 128
    }
}

# ==================== HELPERS ====================
def to_numpy_safe(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.array(x)

def limit_train_size(X_train, y_train, sensitive_dict, max_samples=MAX_TRAIN_SAMPLES, seed=42):
    """Limit training set to max_samples (with replacement if needed)."""
    np.random.seed(seed)
    n = len(X_train)
    if n <= max_samples:
        idx = np.random.choice(n, max_samples, replace=True)
    else:
        idx = np.random.choice(n, max_samples, replace=False)

    X_lim = X_train.iloc[idx].reset_index(drop=True)
    y_lim = y_train[idx]
    s_lim = {k: v.iloc[idx].reset_index(drop=True) for k, v in sensitive_dict.items()}

    print(f"[Data Limit] Train size: {n} → {len(X_lim)} (budget ≤ {max_samples})")
    return X_lim, y_lim, s_lim

def add_z_spur(X, y, correlation='positive', seed=42):
    """Add spurious feature Z_spur with controllable correlation to y."""
    rng = np.random.default_rng(seed)
    mu = np.where(y == 1, 1.0, 0.0)
    if correlation == 'negative':
        mu = -mu
    z = rng.normal(mu, 0.5)
    X_out = X.copy()
    X_out['Z_spur'] = z
    return X_out

def get_preprocessor_with_z(num_cols, cat_cols):
    return ColumnTransformer([
        ('num', StandardScaler(), num_cols + ['Z_spur']),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ])

def evaluate_model(model, X_train, y_train, X_test, y_test, sensitive_dict,
                   is_tabpfn=False, preprocessor=None, eval_cfg=None):
    """Unified evaluation for all models."""
    if is_tabpfn:
        eval_model = clone_model_for_evaluation(model, eval_cfg or {}, TabPFNClassifier)
        eval_model.fit(X_train.values if hasattr(X_train, 'values') else X_train, y_train)
        preds = eval_model.predict(X_test.values if hasattr(X_test, 'values') else X_test)
        probs = eval_model.predict_proba(X_test.values if hasattr(X_test, 'values') else X_test)
        probs = probs[:, 1]
    else:
        X_tr = preprocessor.transform(X_train)
        X_te = preprocessor.transform(X_test)
        model.fit(X_tr, y_train)
        preds = model.predict(X_te)
        probs = model.predict_proba(X_te)[:, 1]

    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)

    fair = {'Sensitive': [], 'DP': [], 'EO': []}
    for name, feat in sensitive_dict.items():
        dp = fm.demographic_parity_difference(y_test, preds, sensitive_features=feat)
        eo = fm.equalized_odds_difference(y_test, preds, sensitive_features=feat)
        fair['Sensitive'].append(name)
        fair['DP'].append(round(dp, 4))
        fair['EO'].append(round(eo, 4))

    return acc, auc, pd.DataFrame(fair), preds

# ==================== MAIN EXPERIMENT ====================
def run_bank_pseudo_correlation_flip():
    # Load data
    df = pd.read_excel('Bank_Personal_Loan_Modelling.xlsx', sheet_name='Data')
    df = df.drop(columns=['ID', 'ZIP Code'], errors='ignore')

    target = "Personal Loan"
    sensitive_cols = {"Education": df["Education"], "Family": df["Family"]}

    X = df.drop(columns=[target])
    y = df[target].values

    num_cols = X.select_dtypes('number').columns.tolist()
    cat_cols = X.select_dtypes('object').columns.tolist()

    results = []

    for run in range(1, 6):
        seed = 41 + run
        np.random.seed(seed)
        torch.manual_seed(seed)
        print(f"\n{'='*30} RUN {run}/5 (seed={seed}) {'='*30}")

        # Train/test split (20% test as in your code)
        X_train_full, X_test, y_train_full, y_test = train_test_split(
            X, y, test_size=0.2, random_state=seed, stratify=y
        )
        s_test = {k: v.iloc[X_test.index] for k, v in sensitive_cols.items()}

        # Limit all models to ≤5000 training samples
        X_train, y_train, s_train = limit_train_size(
            X_train_full, y_train_full,
            {k: v.iloc[X_train_full.index] for k, v in sensitive_cols.items()},
            max_samples=MAX_TRAIN_SAMPLES, seed=seed
        )

        # Add spurious feature
        X_train_z = add_z_spur(X_train, y_train, 'positive', seed=seed)
        X_test_flip = add_z_spur(X_test, y_test, 'negative', seed=seed)
        X_test_normal = add_z_spur(X_test, y_test, 'positive', seed=seed)

        preprocessor = get_preprocessor_with_z(num_cols, cat_cols)
        X_train_prep = preprocessor.fit_transform(X_train_z)

        # Models
        models = {
            'LR': Pipeline([('clf', LogisticRegression(C=0.1, max_iter=1000))]),
            'RF': Pipeline([('clf', RandomForestClassifier(n_estimators=50, max_depth=5))]),
            'MLP': Pipeline([('clf', MLPClassifier(hidden_layer_sizes=(50,), max_iter=300, alpha=0.01))]),
            'TabPFN': TabPFNClassifier(device=config["device"], n_estimators=8),
            'FT-TabPFN': None
        }

        # === Finetune FT-TabPFN ===
        print("Finetuning FT-TabPFN...")
        try:
            clf_cfg = {
                "ignore_pretraining_limits": True,
                "device": config["device"],
                "n_estimators": 1,
                "inference_precision": torch.float32,
            }
            ft_clf = TabPFNClassifier(**clf_cfg, fit_mode="batched", differentiable_input=False)
            ft_clf._initialize_model_variables()

            datasets = ft_clf.get_preprocessed_datasets(
                X_train_prep, y_train,
                partial(train_test_split, test_size=0.3, random_state=seed),
                batch_size=128
            )
            loader = DataLoader(datasets, batch_size=1, collate_fn=meta_dataset_collator, shuffle=True)
            opt = Adam(ft_clf.models_[0].parameters(), lr=config["finetuning"]["learning_rate"])

            for epoch in range(config["finetuning"]["epochs"]):
                for batch in tqdm(loader, desc=f"FT Epoch {epoch+1}", leave=False):
                    X_tr, X_val, y_tr, y_val, cat, conf = batch
                    if len(np.unique(to_numpy_safe(y_tr))) < 2: continue
                    opt.zero_grad()
                    ft_clf.fit_from_preprocessed(X_tr, y_tr, cat, conf)
                    logits = ft_clf.forward(X_val, return_logits=True)
                    loss = torch.nn.CrossEntropyLoss()(logits, y_val.to(config["device"]))
                    loss.backward()
                    opt.step()
            models['FT-TabPFN'] = ft_clf
        except Exception as e:
            print(f"[FT Error] {e}")

        eval_cfg = {"inference_config": {"SUBSAMPLE_SAMPLES": INFERENCE_CONTEXT_SAMPLES}}

        # === Evaluate all models ===
        for name, model in models.items():
            if model is None: continue

            # Flipped test
            acc_f, auc_f, fair_f, preds_f = evaluate_model(
                model, X_train_z, y_train, X_test_flip, y_test, s_test,
                is_tabpfn=name.startswith('TabPFN'),
                preprocessor=preprocessor if not name.startswith('TabPFN') else None,
                eval_cfg=eval_cfg if name == 'FT-TabPFN' else {}
            )
            # Normal test
            acc_n, auc_n, fair_n, preds_n = evaluate_model(
                model, X_train_z, y_train, X_test_normal, y_test, s_test,
                is_tabpfn=name.startswith('TabPFN'),
                preprocessor=preprocessor if not name.startswith('TabPFN') else None,
                eval_cfg=eval_cfg if name == 'FT-TabPFN' else {}
            )

            consistency = np.mean(preds_f == preds_n)
            delta_acc = acc_f - acc_n

            for _, row in fair_f.iterrows():
                s = row['Sensitive']
                dp_f, eo_f = row['DP'], row['EO']
                dp_n = fair_n.loc[fair_n['Sensitive'] == s, 'DP'].iloc[0]
                eo_n = fair_n.loc[fair_n['Sensitive'] == s, 'EO'].iloc[0]

                results.append({
                    'Run': run, 'Model': name, 'Sensitive': s,
                    'Accuracy_Flip': acc_f, 'Accuracy_Normal': acc_n, 'ΔAccuracy': delta_acc,
                    'AUC_Flip': auc_f, 'AUC_Normal': auc_n,
                    'DP_Flip': dp_f, 'DP_Normal': dp_n, 'ΔDP': dp_f - dp_n,
                    'EO_Flip': eo_f, 'EO_Normal': eo_n, 'ΔEO': eo_f - eo_n,
                    'Consistency': consistency
                })

    # === Final Table ===
    df_res = pd.DataFrame(results)
    agg = df_res.groupby(['Model', 'Sensitive']).agg({
        'ΔAccuracy': ['mean', 'std'],
        'ΔDP': ['mean', 'std'],
        'ΔEO': ['mean', 'std'],
        'Consistency': ['mean', 'std'],
        'Accuracy_Normal': 'mean',
        'AUC_Normal': 'mean'
    }).round(4)

    agg.columns = ['ΔAcc_mean', 'ΔAcc_std', 'ΔDP_mean', 'ΔDP_std',
                   'ΔEO_mean', 'ΔEO_std', 'Consistency_mean', 'Consistency_std',
                   'Acc_Base', 'AUC_Base']

    print("\n" + "="*130)
    print("FINAL RESULTS: Bank Personal Loan – All Models Limited to ≤5000 Training Samples")
    print("="*130)
    print(agg.to_markdown())

    return agg

# ==================== RUN ====================
if __name__ == "__main__":
    run_bank_pseudo_correlation_flip()

In [ ]:
"""
Pseudo-Correlation Flip Test for Adult Dataset (N≤5000 per model)
- All models (including LR, RF, MLP) are limited to at most 5000 training samples
- TabPFN & FT-TabPFN use exactly 5000 in-context examples
- Fair comparison under identical data budget
- Sensitive attributes: race, sex
"""

import pandas as pd
import numpy as np
import torch
from torch.optim import Adam
from torch.utils.data import DataLoader
from functools import partial
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
import fairlearn.metrics as fm
from tabpfn import TabPFNClassifier
from tabpfn.utils import meta_dataset_collator
from tabpfn.finetune_utils import clone_model_for_evaluation
import warnings
warnings.filterwarnings("ignore")

# ==================== CONFIG ====================
MAX_TRAIN_SAMPLES = 5000          # 所有模型统一上限
INFERENCE_CONTEXT_SAMPLES = 5000  # TabPFN 推理时强制 5000 条

config = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "n_inference_context_samples": INFERENCE_CONTEXT_SAMPLES,
    "finetuning": {
        "epochs": 10,
        "learning_rate": 1e-5,
        "meta_batch_size": 1,
        "batch_size": 128
    }
}

# ==================== HELPERS ====================
def to_numpy_safe(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.array(x)

def limit_train_size(X_train, y_train, sensitive_dict, max_samples=MAX_TRAIN_SAMPLES, seed=42):
    """Limit training set to max_samples (with replacement if needed)."""
    np.random.seed(seed)
    n = len(X_train)
    if n <= max_samples:
        idx = np.random.choice(n, max_samples, replace=True)
    else:
        idx = np.random.choice(n, max_samples, replace=False)

    X_lim = X_train.iloc[idx].reset_index(drop=True)
    y_lim = y_train[idx]
    s_lim = {k: v.iloc[idx].reset_index(drop=True) for k, v in sensitive_dict.items()}

    print(f"[Data Limit] Train size: {n} to {len(X_lim)} (budget ≤ {max_samples})")
    return X_lim, y_lim, s_lim

def add_z_spur(X, y, correlation='positive', seed=42):
    """Add spurious feature Z_spur with controllable correlation."""
    rng = np.random.default_rng(seed)
    mu = np.where(y == 1, 1.0, 0.0)
    if correlation == 'negative':
        mu = -mu
    z = rng.normal(mu, 0.5)
    X_out = X.copy()
    X_out['Z_spur'] = z
    return X_out

def get_preprocessor_with_z(num_cols, cat_cols):
    return ColumnTransformer([
        ('num', StandardScaler(), num_cols + ['Z_spur']),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ])

def evaluate_model(model, X_train, y_train, X_test, y_test, sensitive_dict,
                   is_tabpfn=False, preprocessor=None, eval_cfg=None):
    """Unified evaluation for all models."""
    if is_tabpfn:
        eval_model = clone_model_for_evaluation(model, eval_cfg or {}, TabPFNClassifier)
        eval_model.fit(X_train.values if hasattr(X_train, 'values') else X_train, y_train)
        preds = eval_model.predict(X_test.values if hasattr(X_test, 'values') else X_test)
        probs = eval_model.predict_proba(X_test.values if hasattr(X_test, 'values') else X_test)
        probs = probs[:, 1]
    else:
        X_tr = preprocessor.transform(X_train)
        X_te = preprocessor.transform(X_test)
        model.fit(X_tr, y_train)
        preds = model.predict(X_te)
        probs = model.predict_proba(X_te)[:, 1]

    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)

    fair = {'Sensitive': [], 'DP': [], 'EO': []}
    for name, feat in sensitive_dict.items():
        dp = fm.demographic_parity_difference(y_test, preds, sensitive_features=feat)
        eo = fm.equalized_odds_difference(y_test, preds, sensitive_features=feat)
        fair['Sensitive'].append(name)
        fair['DP'].append(round(dp, 4))
        fair['EO'].append(round(eo, 4))

    return acc, auc, pd.DataFrame(fair), preds

# ==================== MAIN EXPERIMENT ====================
def run_adult_pseudo_correlation_flip():
    # Load Adult dataset
    df = pd.read_csv('adult.csv')

    target = "income>50K"
    sensitive_cols = {"race": df["race"], "sex": df["sex"]}

    X = df.drop(columns=[target])
    y = (df[target] == 1).astype(int).values  # Ensure binary 0/1

    num_cols = X.select_dtypes('number').columns.tolist()
    cat_cols = X.select_dtypes('object').columns.tolist()

    results = []

    for run in range(1, 6):
        seed = 41 + run
        np.random.seed(seed)
        torch.manual_seed(seed)
        print(f"\n{'='*30} RUN {run}/5 (seed={seed}) {'='*30}")

        # Train/test split (20% test)
        X_train_full, X_test, y_train_full, y_test = train_test_split(
            X, y, test_size=0.2, random_state=seed, stratify=y
        )
        s_test = {k: v.iloc[X_test.index] for k, v in sensitive_cols.items()}

        # Limit all models to ≤5000 training samples
        X_train, y_train, s_train = limit_train_size(
            X_train_full, y_train_full,
            {k: v.iloc[X_train_full.index] for k, v in sensitive_cols.items()},
            max_samples=MAX_TRAIN_SAMPLES, seed=seed
        )

        # Add spurious feature
        X_train_z = add_z_spur(X_train, y_train, 'positive', seed=seed)
        X_test_flip = add_z_spur(X_test, y_test, 'negative', seed=seed)
        X_test_normal = add_z_spur(X_test, y_test, 'positive', seed=seed)

        preprocessor = get_preprocessor_with_z(num_cols, cat_cols)
        X_train_prep = preprocessor.fit_transform(X_train_z)

        # Models
        models = {
            'LR': Pipeline([('clf', LogisticRegression(C=0.1, max_iter=1000))]),
            'RF': Pipeline([('clf', RandomForestClassifier(n_estimators=50, max_depth=5))]),
            'MLP': Pipeline([('clf', MLPClassifier(hidden_layer_sizes=(50,), max_iter=300, alpha=0.01))]),
            'TabPFN': TabPFNClassifier(device=config["device"], n_estimators=8),
            'FT-TabPFN': None
        }

        # === Finetune FT-TabPFN ===
        print("Finetuning FT-TabPFN...")
        try:
            clf_cfg = {
                "ignore_pretraining_limits": True,
                "device": config["device"],
                "n_estimators": 1,
                "inference_precision": torch.float32,
            }
            ft_clf = TabPFNClassifier(**clf_cfg, fit_mode="batched", differentiable_input=False)
            ft_clf._initialize_model_variables()

            datasets = ft_clf.get_preprocessed_datasets(
                X_train_prep, y_train,
                partial(train_test_split, test_size=0.3, random_state=seed),
                batch_size=128
            )
            loader = DataLoader(datasets, batch_size=1, collate_fn=meta_dataset_collator, shuffle=True)
            opt = Adam(ft_clf.models_[0].parameters(), lr=config["finetuning"]["learning_rate"])

            for epoch in range(config["finetuning"]["epochs"]):
                for batch in tqdm(loader, desc=f"FT Epoch {epoch+1}", leave=False):
                    X_tr, X_val, y_tr, y_val, cat, conf = batch
                    if len(np.unique(to_numpy_safe(y_tr))) < 2: continue
                    opt.zero_grad()
                    ft_clf.fit_from_preprocessed(X_tr, y_tr, cat, conf)
                    logits = ft_clf.forward(X_val, return_logits=True)
                    loss = torch.nn.CrossEntropyLoss()(logits, y_val.to(config["device"]))
                    loss.backward()
                    opt.step()
            models['FT-TabPFN'] = ft_clf
        except Exception as e:
            print(f"[FT Error] {e}")

        eval_cfg = {"inference_config": {"SUBSAMPLE_SAMPLES": INFERENCE_CONTEXT_SAMPLES}}

        # === Evaluate all models ===
        for name, model in models.items():
            if model is None: continue

            # Flipped test
            acc_f, auc_f, fair_f, preds_f = evaluate_model(
                model, X_train_z, y_train, X_test_flip, y_test, s_test,
                is_tabpfn=name.startswith('TabPFN'),
                preprocessor=preprocessor if not name.startswith('TabPFN') else None,
                eval_cfg=eval_cfg if name == 'FT-TabPFN' else {}
            )
            # Normal test
            acc_n, auc_n, fair_n, preds_n = evaluate_model(
                model, X_train_z, y_train, X_test_normal, y_test, s_test,
                is_tabpfn=name.startswith('TabPFN'),
                preprocessor=preprocessor if not name.startswith('TabPFN') else None,
                eval_cfg=eval_cfg if name == 'FT-TabPFN' else {}
            )

            consistency = np.mean(preds_f == preds_n)
            delta_acc = acc_f - acc_n

            for _, row in fair_f.iterrows():
                s = row['Sensitive']
                dp_f, eo_f = row['DP'], row['EO']
                dp_n = fair_n.loc[fair_n['Sensitive'] == s, 'DP'].iloc[0]
                eo_n = fair_n.loc[fair_n['Sensitive'] == s, 'EO'].iloc[0]

                results.append({
                    'Run': run, 'Model': name, 'Sensitive': s,
                    'Accuracy_Flip': acc_f, 'Accuracy_Normal': acc_n, 'ΔAccuracy': delta_acc,
                    'AUC_Flip': auc_f, 'AUC_Normal': auc_n,
                    'DP_Flip': dp_f, 'DP_Normal': dp_n, 'ΔDP': dp_f - dp_n,
                    'EO_Flip': eo_f, 'EO_Normal': eo_n, 'ΔEO': eo_f - eo_n,
                    'Consistency': consistency
                })

    # === Final Table ===
    df_res = pd.DataFrame(results)
    agg = df_res.groupby(['Model', 'Sensitive']).agg({
        'ΔAccuracy': ['mean', 'std'],
        'ΔDP': ['mean', 'std'],
        'ΔEO': ['mean', 'std'],
        'Consistency': ['mean', 'std'],
        'Accuracy_Normal': 'mean',
        'AUC_Normal': 'mean'
    }).round(4)

    agg.columns = ['ΔAcc_mean', 'ΔAcc_std', 'ΔDP_mean', 'ΔDP_std',
                   'ΔEO_mean', 'ΔEO_std', 'Consistency_mean', 'Consistency_std',
                   'Acc_Base', 'AUC_Base']

    print("\n" + "="*130)
    print("FINAL RESULTS: Adult – All Models Limited to ≤5000 Training Samples")
    print("="*130)
    print(agg.to_markdown())

    return agg

# ==================== RUN ====================
if __name__ == "__main__":
    run_adult_pseudo_correlation_flip()

In [ ]:
"""
Pseudo-Correlation Flip Test for Law School Dataset (N≤5000 per model)
- All models (including LR, RF, MLP) are limited to at most 5000 training samples
- TabPFN & FT-TabPFN use exactly 5000 in-context examples
- Fair comparison under identical data budget
- Sensitive attributes: race, sex
"""

import pandas as pd
import numpy as np
import torch
from torch.optim import Adam
from torch.utils.data import DataLoader
from functools import partial
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
import fairlearn.metrics as fm
from tabpfn import TabPFNClassifier
from tabpfn.utils import meta_dataset_collator
from tabpfn.finetune_utils import clone_model_for_evaluation
import warnings
warnings.filterwarnings("ignore")

# ==================== CONFIG ====================
MAX_TRAIN_SAMPLES = 5000          # 所有模型统一上限
INFERENCE_CONTEXT_SAMPLES = 5000  # TabPFN 推理时强制 5000 条

config = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "n_inference_context_samples": INFERENCE_CONTEXT_SAMPLES,
    "finetuning": {
        "epochs": 10,
        "learning_rate": 1e-5,
        "meta_batch_size": 1,
        "batch_size": 128
    }
}

# ==================== HELPERS ====================
def to_numpy_safe(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.array(x)

def limit_train_size(X_train, y_train, sensitive_dict, max_samples=MAX_TRAIN_SAMPLES, seed=42):
    """Limit training set to max_samples (with replacement if needed)."""
    np.random.seed(seed)
    n = len(X_train)
    if n <= max_samples:
        idx = np.random.choice(n, max_samples, replace=True)
    else:
        idx = np.random.choice(n, max_samples, replace=False)

    X_lim = X_train.iloc[idx].reset_index(drop=True)
    y_lim = y_train[idx]
    s_lim = {k: v.iloc[idx].reset_index(drop=True) for k, v in sensitive_dict.items()}

    print(f"[Data Limit] Train size: {n} → {len(X_lim)} (budget ≤ {max_samples})")
    return X_lim, y_lim, s_lim

def add_z_spur(X, y, correlation='positive', seed=42):
    """Add spurious feature Z_spur with controllable correlation."""
    rng = np.random.default_rng(seed)
    mu = np.where(y == 1, 1.0, 0.0)
    if correlation == 'negative':
        mu = -mu
    z = rng.normal(mu, 0.5)
    X_out = X.copy()
    X_out['Z_spur'] = z
    return X_out

def get_preprocessor_with_z(num_cols, cat_cols):
    return ColumnTransformer([
        ('num', StandardScaler(), num_cols + ['Z_spur']),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ])

def evaluate_model(model, X_train, y_train, X_test, y_test, sensitive_dict,
                   is_tabpfn=False, preprocessor=None, eval_cfg=None):
    """Unified evaluation for all models."""
    if is_tabpfn:
        eval_model = clone_model_for_evaluation(model, eval_cfg or {}, TabPFNClassifier)
        eval_model.fit(X_train.values if hasattr(X_train, 'values') else X_train, y_train)
        preds = eval_model.predict(X_test.values if hasattr(X_test, 'values') else X_test)
        probs = eval_model.predict_proba(X_test.values if hasattr(X_test, 'values') else X_test)
        probs = probs[:, 1]
    else:
        X_tr = preprocessor.transform(X_train)
        X_te = preprocessor.transform(X_test)
        model.fit(X_tr, y_train)
        preds = model.predict(X_te)
        probs = model.predict_proba(X_te)[:, 1]

    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)

    fair = {'Sensitive': [], 'DP': [], 'EO': []}
    for name, feat in sensitive_dict.items():
        dp = fm.demographic_parity_difference(y_test, preds, sensitive_features=feat)
        eo = fm.equalized_odds_difference(y_test, preds, sensitive_features=feat)
        fair['Sensitive'].append(name)
        fair['DP'].append(round(dp, 4))
        fair['EO'].append(round(eo, 4))

    return acc, auc, pd.DataFrame(fair), preds

# ==================== MAIN EXPERIMENT ====================
def run_law_pseudo_correlation_flip():
    # Load Law School dataset
    df = pd.read_csv('law_data.csv')

    target = "first_pf"
    sensitive_cols = {"race": df["race"], "sex": df["sex"]}

    X = df.drop(columns=[target])
    y = df[target].values

    num_cols = X.select_dtypes('number').columns.tolist()
    cat_cols = X.select_dtypes('object').columns.tolist()

    results = []

    for run in range(1, 6):
        seed = 41 + run
        np.random.seed(seed)
        torch.manual_seed(seed)
        print(f"\n{'='*30} RUN {run}/5 (seed={seed}) {'='*30}")

        # Train/test split (20% test)
        X_train_full, X_test, y_train_full, y_test = train_test_split(
            X, y, test_size=0.2, random_state=seed, stratify=y
        )
        s_test = {k: v.iloc[X_test.index] for k, v in sensitive_cols.items()}

        # Limit all models to ≤5000 training samples
        X_train, y_train, s_train = limit_train_size(
            X_train_full, y_train_full,
            {k: v.iloc[X_train_full.index] for k, v in sensitive_cols.items()},
            max_samples=MAX_TRAIN_SAMPLES, seed=seed
        )

        # Add spurious feature
        X_train_z = add_z_spur(X_train, y_train, 'positive', seed=seed)
        X_test_flip = add_z_spur(X_test, y_test, 'negative', seed=seed)
        X_test_normal = add_z_spur(X_test, y_test, 'positive', seed=seed)

        preprocessor = get_preprocessor_with_z(num_cols, cat_cols)
        X_train_prep = preprocessor.fit_transform(X_train_z)

        # Models
        models = {
            'LR': Pipeline([('clf', LogisticRegression(C=0.1, max_iter=1000))]),
            'RF': Pipeline([('clf', RandomForestClassifier(n_estimators=50, max_depth=5))]),
            'MLP': Pipeline([('clf', MLPClassifier(hidden_layer_sizes=(50,), max_iter=300, alpha=0.01))]),
            'TabPFN': TabPFNClassifier(device=config["device"], n_estimators=8),
            'FT-TabPFN': None
        }

        # === Finetune FT-TabPFN ===
        print("Finetuning FT-TabPFN...")
        try:
            clf_cfg = {
                "ignore_pretraining_limits": True,
                "device": config["device"],
                "n_estimators": 1,
                "inference_precision": torch.float32,
            }
            ft_clf = TabPFNClassifier(**clf_cfg, fit_mode="batched", differentiable_input=False)
            ft_clf._initialize_model_variables()

            datasets = ft_clf.get_preprocessed_datasets(
                X_train_prep, y_train,
                partial(train_test_split, test_size=0.3, random_state=seed),
                batch_size=128
            )
            loader = DataLoader(datasets, batch_size=1, collate_fn=meta_dataset_collator, shuffle=True)
            opt = Adam(ft_clf.models_[0].parameters(), lr=config["finetuning"]["learning_rate"])

            for epoch in range(config["finetuning"]["epochs"]):
                for batch in tqdm(loader, desc=f"FT Epoch {epoch+1}", leave=False):
                    X_tr, X_val, y_tr, y_val, cat, conf = batch
                    if len(np.unique(to_numpy_safe(y_tr))) < 2: continue
                    opt.zero_grad()
                    ft_clf.fit_from_preprocessed(X_tr, y_tr, cat, conf)
                    logits = ft_clf.forward(X_val, return_logits=True)
                    loss = torch.nn.CrossEntropyLoss()(logits, y_val.to(config["device"]))
                    loss.backward()
                    opt.step()
            models['FT-TabPFN'] = ft_clf
        except Exception as e:
            print(f"[FT Error] {e}")

        eval_cfg = {"inference_config": {"SUBSAMPLE_SAMPLES": INFERENCE_CONTEXT_SAMPLES}}

        # === Evaluate all models ===
        for name, model in models.items():
            if model is None: continue

            # Flipped test
            acc_f, auc_f, fair_f, preds_f = evaluate_model(
                model, X_train_z, y_train, X_test_flip, y_test, s_test,
                is_tabpfn=name.startswith('TabPFN'),
                preprocessor=preprocessor if not name.startswith('TabPFN') else None,
                eval_cfg=eval_cfg if name == 'FT-TabPFN' else {}
            )
            # Normal test
            acc_n, auc_n, fair_n, preds_n = evaluate_model(
                model, X_train_z, y_train, X_test_normal, y_test, s_test,
                is_tabpfn=name.startswith('TabPFN'),
                preprocessor=preprocessor if not name.startswith('TabPFN') else None,
                eval_cfg=eval_cfg if name == 'FT-TabPFN' else {}
            )

            consistency = np.mean(preds_f == preds_n)
            delta_acc = acc_f - acc_n

            for _, row in fair_f.iterrows():
                s = row['Sensitive']
                dp_f, eo_f = row['DP'], row['EO']
                dp_n = fair_n.loc[fair_n['Sensitive'] == s, 'DP'].iloc[0]
                eo_n = fair_n.loc[fair_n['Sensitive'] == s, 'EO'].iloc[0]

                results.append({
                    'Run': run, 'Model': name, 'Sensitive': s,
                    'Accuracy_Flip': acc_f, 'Accuracy_Normal': acc_n, 'ΔAccuracy': delta_acc,
                    'AUC_Flip': auc_f, 'AUC_Normal': auc_n,
                    'DP_Flip': dp_f, 'DP_Normal': dp_n, 'ΔDP': dp_f - dp_n,
                    'EO_Flip': eo_f, 'EO_Normal': eo_n, 'ΔEO': eo_f - eo_n,
                    'Consistency': consistency
                })

    # === Final Table ===
    df_res = pd.DataFrame(results)
    agg = df_res.groupby(['Model', 'Sensitive']).agg({
        'ΔAccuracy': ['mean', 'std'],
        'ΔDP': ['mean', 'std'],
        'ΔEO': ['mean', 'std'],
        'Consistency': ['mean', 'std'],
        'Accuracy_Normal': 'mean',
        'AUC_Normal': 'mean'
    }).round(4)

    agg.columns = ['ΔAcc_mean', 'ΔAcc_std', 'ΔDP_mean', 'ΔDP_std',
                   'ΔEO_mean', 'ΔEO_std', 'Consistency_mean', 'Consistency_std',
                   'Acc_Base', 'AUC_Base']

    print("\n" + "="*130)
    print("FINAL RESULTS: Law School – All Models Limited to ≤5000 Training Samples")
    print("="*130)
    print(agg.to_markdown())

    return agg

# ==================== RUN ====================
if __name__ == "__main__":
    run_law_pseudo_correlation_flip()